# Python Generators

## Learning Objectives
- Understand what a **generator** is and how it differs from a regular function or list
- Know the two ways to create generators: **generator functions** (`yield`) and **generator expressions**
- Understand the generator protocol: `__iter__`, `__next__`, `send`, `throw`, `close`
- Recognise when generators give a **memory and performance advantage** over materialised collections
- Know the **limitations** of generators: single-pass, no random access, no `len`

## What is a Generator?

A **generator** is a Python object that produces values **one at a time, on demand** (lazy evaluation).  
Unlike a list, it does not compute or store all values upfront — each value is produced only when the caller asks for it with `next()`.

Every generator is an **iterator**, meaning it implements the iterator protocol:

| Method | What it does |
|---|---|
| `__iter__()` | Returns the generator itself |
| `__next__()` | Resumes execution and returns the next yielded value; raises `StopIteration` when done |

### Mental model — a paused function

A generator function is like a function whose execution can be **paused** at a `yield` statement and **resumed** later, preserving its entire local state (variables, instruction pointer) between calls.

In [14]:
from IPython.display import HTML, display
display(HTML("""
<figure style="margin:16px 0">
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 560 230" width="560" height="230"
     font-family="Segoe UI, Arial, sans-serif">
  <defs>
    <marker id="mm1" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto">
      <path d="M0,0 L0,6 L8,3 z" fill="#2166ac"/>
    </marker>
    <marker id="mm2" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto">
      <path d="M0,0 L0,6 L8,3 z" fill="#3a8c3f"/>
    </marker>
    <marker id="mm3" markerWidth="8" markerHeight="8" refX="6" refY="3" orient="auto">
      <path d="M0,0 L0,6 L8,3 z" fill="#c0392b"/>
    </marker>
  </defs>
  <rect width="560" height="230" fill="#f9f9fb" rx="8"/>
  <rect x="16" y="14" width="110" height="30" rx="5" fill="#2166ac"/>
  <text x="71" y="33" text-anchor="middle" font-size="12" font-weight="bold" fill="white">Caller</text>
  <rect x="434" y="14" width="110" height="30" rx="5" fill="#c0392b"/>
  <text x="489" y="33" text-anchor="middle" font-size="12" font-weight="bold" fill="white">Generator</text>
  <line x1="71" y1="44" x2="71" y2="222" stroke="#2166ac" stroke-width="1.5" stroke-dasharray="6,4" opacity="0.3"/>
  <line x1="489" y1="44" x2="489" y2="222" stroke="#c0392b" stroke-width="1.5" stroke-dasharray="6,4" opacity="0.3"/>
  <line x1="73" y1="72" x2="432" y2="72" stroke="#2166ac" stroke-width="1.6" marker-end="url(#mm1)"/>
  <text x="252" y="67" text-anchor="middle" font-size="10" fill="#2166ac">next()</text>
  <text x="489" y="84" text-anchor="middle" font-size="9" fill="#555">resumes from last yield</text>
  <text x="489" y="96" text-anchor="middle" font-size="9" fill="#555">... executes ...</text>
  <line x1="432" y1="108" x2="73" y2="108" stroke="#3a8c3f" stroke-width="1.6" marker-end="url(#mm2)"/>
  <text x="252" y="103" text-anchor="middle" font-size="10" fill="#3a8c3f">value</text>
  <text x="489" y="120" text-anchor="middle" font-size="9" fill="#555">pauses at next yield</text>
  <line x1="100" y1="132" x2="460" y2="132" stroke="#ddd" stroke-width="1" stroke-dasharray="3,4"/>
  <line x1="73" y1="152" x2="432" y2="152" stroke="#2166ac" stroke-width="1.6" marker-end="url(#mm1)"/>
  <text x="252" y="147" text-anchor="middle" font-size="10" fill="#2166ac">next()</text>
  <text x="489" y="164" text-anchor="middle" font-size="9" fill="#555">resumes again</text>
  <text x="489" y="176" text-anchor="middle" font-size="9" fill="#555">... executes ...</text>
  <line x1="432" y1="190" x2="73" y2="190" stroke="#c0392b" stroke-width="1.6" stroke-dasharray="5,3" marker-end="url(#mm3)"/>
  <text x="252" y="185" text-anchor="middle" font-size="10" fill="#c0392b">StopIteration</text>
  <text x="489" y="202" text-anchor="middle" font-size="9" fill="#555">function body ends</text>
  <rect x="148" y="210" width="264" height="16" rx="4" fill="#fff3cd"/>
  <text x="280" y="222" text-anchor="middle" font-size="9" fill="#856404">Local variables &amp; instruction pointer preserved between calls</text>
</svg>
<figcaption style="font-size:0.82em;color:#555;margin-top:4px;text-align:center">
  Solid arrows = data; dashed red = exception. The generator stack frame is frozen between <code>yield</code> and the next <code>next()</code>.
</figcaption>
</figure>
"""))

In [15]:
from IPython.display import HTML, display
display(HTML("""
<figure style="margin:16px 0">
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 740 430" width="740" height="430"
     font-family="Segoe UI, Arial, sans-serif">
  <defs>
    <marker id="mBlue" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto">
      <path d="M0,0 L0,6 L9,3 z" fill="#2166ac"/>
    </marker>
    <marker id="mGreen" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto">
      <path d="M0,0 L0,6 L9,3 z" fill="#3a8c3f"/>
    </marker>
    <marker id="mRed" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto">
      <path d="M0,0 L0,6 L9,3 z" fill="#c0392b"/>
    </marker>
    <marker id="mGray" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto">
      <path d="M0,0 L0,6 L9,3 z" fill="#555"/>
    </marker>
  </defs>
  <rect width="740" height="430" fill="#f9f9fb" rx="10"/>
  <text x="370" y="32" text-anchor="middle" font-size="15" font-weight="bold" fill="#1a1a2e">Execution Flow &#8212; Caller &#8596; Generator</text>
  <rect x="20"  y="48" width="150" height="36" rx="6" fill="#2166ac"/>
  <text x="95"  y="71" text-anchor="middle" font-size="13" font-weight="bold" fill="white">Caller</text>
  <rect x="570" y="48" width="150" height="36" rx="6" fill="#c0392b"/>
  <text x="645" y="71" text-anchor="middle" font-size="13" font-weight="bold" fill="white">Generator</text>
  <line x1="95"  y1="84" x2="95"  y2="420" stroke="#2166ac" stroke-width="2" stroke-dasharray="7,4" opacity="0.35"/>
  <line x1="645" y1="84" x2="645" y2="420" stroke="#c0392b" stroke-width="2" stroke-dasharray="7,4" opacity="0.35"/>
  <rect x="20" y="105" width="150" height="28" rx="5" fill="#dce9f7"/>
  <text x="95" y="123" text-anchor="middle" font-size="10.5" fill="#2166ac" font-weight="bold">gen = countdown(3)</text>
  <line x1="172" y1="119" x2="568" y2="119" stroke="#555" stroke-width="1.6" marker-end="url(#mGray)"/>
  <text x="370" y="114" text-anchor="middle" font-size="10" fill="#555">call &#8212; body does NOT run yet</text>
  <rect x="570" y="105" width="150" height="28" rx="5" fill="#fde8e8"/>
  <text x="645" y="123" text-anchor="middle" font-size="10.5" fill="#c0392b">generator object created</text>
  <rect x="20" y="158" width="150" height="28" rx="5" fill="#dce9f7"/>
  <text x="95" y="176" text-anchor="middle" font-size="10.5" fill="#2166ac" font-weight="bold">next(gen)</text>
  <line x1="172" y1="172" x2="568" y2="172" stroke="#2166ac" stroke-width="1.8" marker-end="url(#mBlue)"/>
  <text x="370" y="167" text-anchor="middle" font-size="10" fill="#2166ac">resume execution</text>
  <rect x="570" y="158" width="150" height="44" rx="5" fill="#e8f5e9"/>
  <text x="645" y="177" text-anchor="middle" font-size="10" fill="#2e7d32">runs body&#8230;</text>
  <text x="645" y="193" text-anchor="middle" font-size="10" fill="#2e7d32" font-weight="bold">yield 3 &#8594; pause</text>
  <line x1="568" y1="210" x2="172" y2="210" stroke="#3a8c3f" stroke-width="1.8" marker-end="url(#mGreen)"/>
  <text x="370" y="205" text-anchor="middle" font-size="10" fill="#3a8c3f" font-weight="bold">value = 3</text>
  <rect x="20" y="214" width="150" height="24" rx="5" fill="#e8f5e9"/>
  <text x="95" y="230" text-anchor="middle" font-size="10" fill="#2e7d32">receives 3 &#10003;</text>
  <rect x="20" y="255" width="150" height="28" rx="5" fill="#dce9f7"/>
  <text x="95" y="273" text-anchor="middle" font-size="10.5" fill="#2166ac" font-weight="bold">next(gen)</text>
  <line x1="172" y1="269" x2="568" y2="269" stroke="#2166ac" stroke-width="1.8" marker-end="url(#mBlue)"/>
  <text x="370" y="264" text-anchor="middle" font-size="10" fill="#2166ac">resume from last yield</text>
  <rect x="570" y="255" width="150" height="44" rx="5" fill="#e8f5e9"/>
  <text x="645" y="274" text-anchor="middle" font-size="10" fill="#2e7d32">resumes&#8230; n=2</text>
  <text x="645" y="290" text-anchor="middle" font-size="10" fill="#2e7d32" font-weight="bold">yield 2 &#8594; pause</text>
  <line x1="568" y1="307" x2="172" y2="307" stroke="#3a8c3f" stroke-width="1.8" marker-end="url(#mGreen)"/>
  <text x="370" y="302" text-anchor="middle" font-size="10" fill="#3a8c3f" font-weight="bold">value = 2</text>
  <rect x="20" y="311" width="150" height="24" rx="5" fill="#e8f5e9"/>
  <text x="95" y="327" text-anchor="middle" font-size="10" fill="#2e7d32">receives 2 &#10003;</text>
  <rect x="20" y="353" width="150" height="28" rx="5" fill="#dce9f7"/>
  <text x="95" y="371" text-anchor="middle" font-size="10.5" fill="#2166ac" font-weight="bold">next(gen)</text>
  <line x1="172" y1="367" x2="568" y2="367" stroke="#2166ac" stroke-width="1.8" marker-end="url(#mBlue)"/>
  <text x="370" y="362" text-anchor="middle" font-size="10" fill="#2166ac">resume</text>
  <rect x="570" y="353" width="150" height="44" rx="5" fill="#fde8e8"/>
  <text x="645" y="372" text-anchor="middle" font-size="10" fill="#c0392b">runs&#8230; n=0</text>
  <text x="645" y="388" text-anchor="middle" font-size="10" fill="#c0392b" font-weight="bold">return &#8594; exhausted</text>
  <line x1="568" y1="403" x2="172" y2="403" stroke="#c0392b" stroke-width="1.8" stroke-dasharray="6,3" marker-end="url(#mRed)"/>
  <text x="370" y="398" text-anchor="middle" font-size="10" fill="#c0392b" font-weight="bold">StopIteration &#10007;</text>
</svg>
<figcaption style="font-size:0.85em;color:#555;margin-top:4px;text-align:center">
  Each <code>next()</code> resumes the generator at the last <code>yield</code>; local state is preserved between calls.
</figcaption>
</figure>
"""))

## How to Implement

### Method 1 — Generator Function (`yield`)

Replace `return` with `yield`.  The presence of any `yield` in a function body makes it a **generator function** — calling it returns a generator object without executing any of the body.

```python
def gen_func(...):
    ...
    yield value       # pause, emit value
    ...
    yield value2      # pause again
    # implicit StopIteration when the function returns
```

Key points:
- Calling `gen_func()` returns a **generator object** immediately; no body code runs yet
- Each `next()` call runs the body until the next `yield`, pauses, and hands the value to the caller
- When the function returns (or falls off the end), `StopIteration` is raised automatically

In [16]:
# Generator function: count up to n
def countdown(n):
    print(f"[generator] starting — will count from {n} to 1")
    while n > 0:
        print(f"[generator] about to yield {n}")
        yield n
        print(f"[generator] resumed after yielding {n}")
        n -= 1
    print("[generator] done")


gen = countdown(3)           # body does NOT run yet
print("Generator object created:", gen)
print()

print("Calling next() #1:", next(gen))
print()
print("Calling next() #2:", next(gen))
print()
print("Calling next() #3:", next(gen))
print()

try:
    print("Calling next() #4 (exhausted):", next(gen))
except StopIteration:
    print("StopIteration raised — generator is exhausted")

Generator object created: <generator object countdown at 0x792640543c30>

[generator] starting — will count from 3 to 1
[generator] about to yield 3
Calling next() #1: 3

[generator] resumed after yielding 3
[generator] about to yield 2
Calling next() #2: 2

[generator] resumed after yielding 2
[generator] about to yield 1
Calling next() #3: 1

[generator] resumed after yielding 1
[generator] done
StopIteration raised — generator is exhausted


In [17]:
# Generators work transparently in for-loops (for calls next() until StopIteration)
def squares(n):
    for i in range(1, n + 1):
        yield i * i


print("Squares via for-loop:")
for sq in squares(6):
    print(sq, end="  ")

print("\n")

# And in any iterable context
total = sum(squares(100))
print(f"Sum of first 100 squares: {total}")

as_list = list(squares(5))
print(f"Materialised to list: {as_list}")

Squares via for-loop:
1  4  9  16  25  36  

Sum of first 100 squares: 338350
Materialised to list: [1, 4, 9, 16, 25]


In [18]:
# yield from — delegate to a sub-generator (Python 3.3+)
def chain(*iterables):
    for it in iterables:
        yield from it          # flattens any iterable inline


print(list(chain([1, 2], [3, 4], [5])))
print(list(chain("ab", "cd", "ef")))

[1, 2, 3, 4, 5]
['a', 'b', 'c', 'd', 'e', 'f']


### Method 2 — Generator Expression

A generator expression is the **lazy equivalent of a list comprehension** — same syntax but with `()` instead of `[]`.

```python
gen_expr  = (expr for item in iterable if condition)   # generator — lazy
list_comp = [expr for item in iterable if condition]   # list — eager
```

Use a generator expression when you only need to **iterate once** and do not need random access or `len`.

In [19]:
import sys

n = 1_000_000

list_comp = [x * x for x in range(n)]
gen_expr  = (x * x for x in range(n))

print(f"List comprehension size : {sys.getsizeof(list_comp):>12,} bytes")
print(f"Generator expression size: {sys.getsizeof(gen_expr):>12,} bytes")
print()

# Generator expression passed directly to a function — no extra parentheses needed
total = sum(x * x for x in range(n))
print(f"Sum of first {n:,} squares: {total:,}")

List comprehension size :    8,448,728 bytes
Generator expression size:          104 bytes

Sum of first 1,000,000 squares: 333,332,833,333,500,000


In [20]:
from IPython.display import HTML, display
display(HTML("""
<figure style="margin:16px 0">
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 740 280" width="740" height="280"
     font-family="Segoe UI, Arial, sans-serif">
  <defs>
    <marker id="ma2" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto">
      <path d="M0,0 L0,6 L9,3 z" fill="#3a8c3f"/>
    </marker>
  </defs>
  <rect width="740" height="280" fill="#f9f9fb" rx="10"/>
  <text x="370" y="28" text-anchor="middle" font-size="15" font-weight="bold" fill="#1a1a2e">Memory Layout &#8212; List vs Generator</text>
  <text x="185" y="58" text-anchor="middle" font-size="13" font-weight="bold" fill="#c0392b">List  [x*x for x in range(n)]</text>
  <rect x="20" y="68" width="330" height="130" rx="8" fill="#fde8e8" stroke="#c0392b" stroke-width="1.5" stroke-dasharray="6,3"/>
  <text x="35" y="86" font-size="10" fill="#c0392b" font-style="italic">RAM &#8212; holds ALL n objects</text>
  <rect x="30"  y="92" width="42" height="36" rx="4" fill="#e8a0a0" stroke="#c0392b" stroke-width="1.2"/>
  <text x="51"  y="116" text-anchor="middle" font-size="12" fill="#7b0000">0</text>
  <rect x="76"  y="92" width="42" height="36" rx="4" fill="#e8a0a0" stroke="#c0392b" stroke-width="1.2"/>
  <text x="97"  y="116" text-anchor="middle" font-size="12" fill="#7b0000">1</text>
  <rect x="122" y="92" width="42" height="36" rx="4" fill="#e8a0a0" stroke="#c0392b" stroke-width="1.2"/>
  <text x="143" y="116" text-anchor="middle" font-size="12" fill="#7b0000">4</text>
  <rect x="168" y="92" width="42" height="36" rx="4" fill="#e8a0a0" stroke="#c0392b" stroke-width="1.2"/>
  <text x="189" y="116" text-anchor="middle" font-size="12" fill="#7b0000">9</text>
  <rect x="214" y="92" width="42" height="36" rx="4" fill="#e8a0a0" stroke="#c0392b" stroke-width="1.2"/>
  <text x="235" y="116" text-anchor="middle" font-size="12" fill="#7b0000">16</text>
  <text x="272" y="116" text-anchor="middle" font-size="18" fill="#c0392b">&#8230;</text>
  <rect x="290" y="92" width="50" height="36" rx="4" fill="#e8a0a0" stroke="#c0392b" stroke-width="1.2"/>
  <text x="315" y="110" text-anchor="middle" font-size="9" fill="#7b0000">(n-1)&#178;</text>
  <text x="185" y="150" text-anchor="middle" font-size="10" fill="#c0392b">O(n) memory &#8212; all values allocated upfront</text>
  <text x="185" y="165" text-anchor="middle" font-size="10" fill="#c0392b">random access &#10003;   len() &#10003;   reusable &#10003;</text>
  <text x="555" y="58" text-anchor="middle" font-size="13" font-weight="bold" fill="#2e7d32">Generator  (x*x for x in range(n))</text>
  <rect x="390" y="68" width="330" height="130" rx="8" fill="#e8f5e9" stroke="#3a8c3f" stroke-width="1.5" stroke-dasharray="6,3"/>
  <text x="405" y="86" font-size="10" fill="#3a8c3f" font-style="italic">RAM &#8212; only generator state object</text>
  <rect x="430" y="92" width="220" height="68" rx="6" fill="#c8e6c9" stroke="#3a8c3f" stroke-width="1.5"/>
  <text x="540" y="112" text-anchor="middle" font-size="11" font-weight="bold" fill="#1b5e20">Generator State</text>
  <text x="540" y="128" text-anchor="middle" font-size="10" fill="#1b5e20">current  x = 3   (just yielded 9)</text>
  <text x="540" y="143" text-anchor="middle" font-size="10" fill="#1b5e20">instruction pointer + local vars</text>
  <line x1="652" y1="118" x2="700" y2="118" stroke="#3a8c3f" stroke-width="1.8" marker-end="url(#ma2)"/>
  <rect x="702" y="105" width="14" height="26" rx="3" fill="#3a8c3f"/>
  <text x="709" y="122" text-anchor="middle" font-size="9" fill="white">9</text>
  <text x="555" y="150" text-anchor="middle" font-size="10" fill="#3a8c3f">O(1) memory &#8212; one value at a time</text>
  <text x="555" y="165" text-anchor="middle" font-size="10" fill="#3a8c3f">random access &#10007;   len() &#10007;   single-pass &#10007;</text>
  <line x1="370" y1="60" x2="370" y2="210" stroke="#ccc" stroke-width="1.5" stroke-dasharray="4,3"/>
  <text x="370" y="228" text-anchor="middle" font-size="11" font-weight="bold" fill="#333">Memory footprint for n = 1 000 000</text>
  <rect x="20"  y="238" width="280" height="20" rx="4" fill="#e8a0a0" stroke="#c0392b" stroke-width="1"/>
  <text x="160" y="252" text-anchor="middle" font-size="10" fill="#7b0000">List  &#8776; 8.5 MB</text>
  <rect x="390" y="238" width="8"   height="20" rx="2" fill="#3a8c3f"/>
  <text x="405" y="252" text-anchor="start"  font-size="10" fill="#1b5e20">Generator  &#8776; 112 bytes</text>
</svg>
<figcaption style="font-size:0.85em;color:#555;margin-top:4px;text-align:center">
  The generator object is fixed-size regardless of how many values it can produce.
</figcaption>
</figure>
"""))

### Method 3 — Generator Protocol (`send`, `throw`, `close`)

A generator is a **coroutine-lite**: the caller can not only pull values out via `next()` but also **push values in** via `send()`.  
This enables two-way communication between the caller and the generator body.

| Operation | Behaviour |
|---|---|
| `next(gen)` | Resume; `yield` evaluates to `None` inside the generator |
| `gen.send(value)` | Resume; `yield` evaluates to `value` inside the generator |
| `gen.throw(exc)` | Raise `exc` at the current `yield` point |
| `gen.close()` | Throw `GeneratorExit`; lets the generator do cleanup |

In [21]:
from IPython.display import HTML, display
display(HTML("""
<figure style="margin:16px 0">
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 720 310" width="720" height="310"
     font-family="Segoe UI, Arial, sans-serif">
  <defs>
    <marker id="sm1" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto">
      <path d="M0,0 L0,6 L9,3 z" fill="#555"/>
    </marker>
    <marker id="sm2" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto">
      <path d="M0,0 L0,6 L9,3 z" fill="#2166ac"/>
    </marker>
    <marker id="sm3" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto">
      <path d="M0,0 L0,6 L9,3 z" fill="#3a8c3f"/>
    </marker>
    <marker id="sm4" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto">
      <path d="M0,0 L0,6 L9,3 z" fill="#c0392b"/>
    </marker>
    <marker id="sm5" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto">
      <path d="M0,0 L0,6 L9,3 z" fill="#7b2d8b"/>
    </marker>
  </defs>
  <rect width="720" height="310" fill="#f9f9fb" rx="10"/>
  <text x="360" y="28" text-anchor="middle" font-size="15" font-weight="bold" fill="#1a1a2e">Generator State Machine</text>
  <rect x="20"  y="110" width="120" height="52" rx="10" fill="#e8ecf5" stroke="#555" stroke-width="1.8"/>
  <text x="80"  y="133" text-anchor="middle" font-size="12" font-weight="bold" fill="#333">Created</text>
  <text x="80"  y="150" text-anchor="middle" font-size="10" fill="#555">gen = f()</text>
  <rect x="290" y="50"  width="140" height="52" rx="10" fill="#dce9f7" stroke="#2166ac" stroke-width="2"/>
  <text x="360" y="73"  text-anchor="middle" font-size="12" font-weight="bold" fill="#2166ac">Suspended</text>
  <text x="360" y="90"  text-anchor="middle" font-size="10" fill="#2166ac">paused at yield</text>
  <rect x="290" y="165" width="140" height="52" rx="10" fill="#e8f5e9" stroke="#3a8c3f" stroke-width="2"/>
  <text x="360" y="188" text-anchor="middle" font-size="12" font-weight="bold" fill="#2e7d32">Running</text>
  <text x="360" y="205" text-anchor="middle" font-size="10" fill="#2e7d32">executing body</text>
  <rect x="580" y="110" width="120" height="52" rx="10" fill="#fde8e8" stroke="#c0392b" stroke-width="1.8"/>
  <text x="640" y="131" text-anchor="middle" font-size="12" font-weight="bold" fill="#c0392b">Closed</text>
  <text x="640" y="148" text-anchor="middle" font-size="10" fill="#c0392b">exhausted / closed</text>
  <path d="M140,148 Q215,200 288,200" fill="none" stroke="#555" stroke-width="1.8" marker-end="url(#sm1)"/>
  <text x="205" y="195" text-anchor="middle" font-size="10" fill="#555">next() / send(None)</text>
  <path d="M360,165 L360,104" fill="none" stroke="#2166ac" stroke-width="1.8" marker-end="url(#sm2)"/>
  <text x="392" y="138" text-anchor="start" font-size="10" fill="#2166ac">yield value</text>
  <path d="M330,165 Q310,145 330,122" fill="none" stroke="#3a8c3f" stroke-width="1.8" marker-end="url(#sm3)"/>
  <text x="240" y="148" text-anchor="middle" font-size="10" fill="#3a8c3f">next() /</text>
  <text x="240" y="160" text-anchor="middle" font-size="10" fill="#3a8c3f">send(val)</text>
  <path d="M430,200 Q510,220 578,155" fill="none" stroke="#c0392b" stroke-width="1.8" marker-end="url(#sm4)"/>
  <text x="520" y="220" text-anchor="middle" font-size="10" fill="#c0392b">return /</text>
  <text x="520" y="232" text-anchor="middle" font-size="10" fill="#c0392b">StopIteration</text>
  <path d="M430,76 Q510,80 578,122" fill="none" stroke="#7b2d8b" stroke-width="1.8" stroke-dasharray="5,3" marker-end="url(#sm5)"/>
  <text x="510" y="74" text-anchor="middle" font-size="10" fill="#7b2d8b">close() /</text>
  <text x="510" y="86" text-anchor="middle" font-size="10" fill="#7b2d8b">throw(exc)</text>
  <line x1="30"  y1="276" x2="60"  y2="276" stroke="#555" stroke-width="1.8" marker-end="url(#sm1)"/>
  <text x="65"  y="280" font-size="10" fill="#555">first call</text>
  <line x1="150" y1="276" x2="180" y2="276" stroke="#3a8c3f" stroke-width="1.8" marker-end="url(#sm3)"/>
  <text x="185" y="280" font-size="10" fill="#3a8c3f">resume (next / send)</text>
  <line x1="330" y1="276" x2="360" y2="276" stroke="#2166ac" stroke-width="1.8" marker-end="url(#sm2)"/>
  <text x="365" y="280" font-size="10" fill="#2166ac">yield</text>
  <line x1="420" y1="276" x2="450" y2="276" stroke="#c0392b" stroke-width="1.8" marker-end="url(#sm4)"/>
  <text x="455" y="280" font-size="10" fill="#c0392b">return / exhaust</text>
  <line x1="570" y1="276" x2="600" y2="276" stroke="#7b2d8b" stroke-width="1.8" stroke-dasharray="5,3" marker-end="url(#sm5)"/>
  <text x="605" y="280" font-size="10" fill="#7b2d8b">close / throw</text>
</svg>
<figcaption style="font-size:0.85em;color:#555;margin-top:4px;text-align:center">
  <code>send(val)</code> is like <code>next()</code> but passes <code>val</code> back as the result of the current <code>yield</code>.
</figcaption>
</figure>
"""))

In [22]:
# Running accumulator — caller sends new values in, generator yields running total
def running_total():
    total = 0
    while True:
        value = yield total      # yield sends total out; receives next value via send()
        if value is None:
            break
        total += value


acc = running_total()
next(acc)            # advance to first yield (required before send)

for v in [10, 20, 5, 100]:
    total = acc.send(v)
    print(f"Sent {v:>4}  →  running total = {total}")

acc.close()          # generator cleaned up

Sent   10  →  running total = 10
Sent   20  →  running total = 30
Sent    5  →  running total = 35
Sent  100  →  running total = 135


In [23]:
# Infinite generator — generators can run forever; caller controls when to stop
def fibonacci():
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b


def take(n, gen):
    return [next(gen) for _ in range(n)]


fib = fibonacci()
print("First 15 Fibonacci numbers:", take(15, fib))

First 15 Fibonacci numbers: [0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377]


## Advantages

| # | Advantage | Why it matters |
|---|---|---|
| 1 | **Memory efficiency** | Only one value lives in memory at a time; a generator over 1 billion items uses the same memory as one over 10 |
| 2 | **Lazy evaluation** | Values are computed only when needed; expensive work is deferred or skipped entirely |
| 3 | **Infinite sequences** | Can model sequences with no natural end (streams, sensor data, Fibonacci) without special-casing |
| 4 | **Pipelining** | Generators can be chained: each stage consumes the previous lazily, keeping the pipeline memory-constant regardless of data size |
| 5 | **Clean state management** | Local variables and the instruction pointer are preserved automatically — no need to manage state in a class or closure |
| 6 | **Simpler code** | Complex iterators that require `__iter__`/`__next__` in a class can often be expressed in a few lines with `yield` |

In [24]:
import sys
import time

# Advantage 1 & 2 — memory and lazy evaluation
N = 10_000_000

t0 = time.perf_counter()
list_result = sum([x for x in range(N)])     # builds the full list first
t_list = time.perf_counter() - t0

t0 = time.perf_counter()
gen_result = sum(x for x in range(N))        # pulls one element at a time
t_gen = time.perf_counter() - t0

print(f"List comprehension  — time: {t_list:.3f}s  |  same result: {list_result == gen_result}")
print(f"Generator expression— time: {t_gen:.3f}s  |  same result: {list_result == gen_result}")
print()

# Memory: generator is essentially fixed-size regardless of N
small_gen  = (x for x in range(10))
large_gen  = (x for x in range(N))
small_list = list(range(10))
large_list = list(range(N))

print(f"Generator (10 items)  : {sys.getsizeof(small_gen):>12,} bytes")
print(f"Generator ({N:,} items): {sys.getsizeof(large_gen):>12,} bytes")
print(f"List      (10 items)  : {sys.getsizeof(small_list):>12,} bytes")
print(f"List      ({N:,} items): {sys.getsizeof(large_list):>12,} bytes")

List comprehension  — time: 0.563s  |  same result: True
Generator expression— time: 0.247s  |  same result: True

Generator (10 items)  :          104 bytes
Generator (10,000,000 items):          104 bytes
List      (10 items)  :          136 bytes
List      (10,000,000 items):   80,000,056 bytes


In [25]:
# Advantage 3 & 4 — infinite sequences + pipelining

def integers_from(start=0):
    n = start
    while True:
        yield n
        n += 1


def filter_gen(pred, gen):
    for v in gen:
        if pred(v):
            yield v


def map_gen(func, gen):
    for v in gen:
        yield func(v)


# Pipeline: integers → only odd → squared → first 8
pipeline = map_gen(lambda x: x * x,
           filter_gen(lambda x: x % 2 != 0,
           integers_from(1)))

result = take(8, pipeline)
print("First 8 odd squares via generator pipeline:", result)
print("Memory stays constant regardless of how many items the pipeline could produce")

First 8 odd squares via generator pipeline: [1, 9, 25, 49, 81, 121, 169, 225]
Memory stays constant regardless of how many items the pipeline could produce


In [26]:
from IPython.display import HTML, display
display(HTML("""
<figure style="margin:16px 0">
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 740 260" width="740" height="260"
     font-family="Segoe UI, Arial, sans-serif">
  <defs>
    <marker id="pp1" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto">
      <path d="M0,0 L0,6 L9,3 z" fill="#3a8c3f"/>
    </marker>
    <marker id="pp2" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto">
      <path d="M0,0 L0,6 L9,3 z" fill="#c0392b"/>
    </marker>
  </defs>
  <rect width="740" height="260" fill="#f9f9fb" rx="10"/>
  <text x="370" y="28" text-anchor="middle" font-size="15" font-weight="bold" fill="#1a1a2e">Generator Pipeline &#8212; Lazy Pull Chain</text>
  <rect x="16"  y="80" width="130" height="70" rx="8" fill="#e8ecf5" stroke="#555" stroke-width="1.8"/>
  <text x="81"  y="106" text-anchor="middle" font-size="11" font-weight="bold" fill="#333">Source</text>
  <text x="81"  y="122" text-anchor="middle" font-size="10" fill="#555">integers_from(0)</text>
  <text x="81"  y="138" text-anchor="middle" font-size="9"  fill="#555">0, 1, 2, 3, 4, 5 &#8230;</text>
  <rect x="196" y="80" width="130" height="70" rx="8" fill="#dce9f7" stroke="#2166ac" stroke-width="1.8"/>
  <text x="261" y="106" text-anchor="middle" font-size="11" font-weight="bold" fill="#2166ac">Filter</text>
  <text x="261" y="122" text-anchor="middle" font-size="10" fill="#2166ac">odd only</text>
  <text x="261" y="138" text-anchor="middle" font-size="9"  fill="#2166ac">1, 3, 5, 7, 9 &#8230;</text>
  <rect x="376" y="80" width="130" height="70" rx="8" fill="#e8f5e9" stroke="#3a8c3f" stroke-width="1.8"/>
  <text x="441" y="106" text-anchor="middle" font-size="11" font-weight="bold" fill="#2e7d32">Map</text>
  <text x="441" y="122" text-anchor="middle" font-size="10" fill="#2e7d32">x &#8594; x&#178;</text>
  <text x="441" y="138" text-anchor="middle" font-size="9"  fill="#2e7d32">1, 9, 25, 49, 81 &#8230;</text>
  <rect x="556" y="80" width="168" height="70" rx="8" fill="#fff3cd" stroke="#e6a817" stroke-width="1.8"/>
  <text x="640" y="106" text-anchor="middle" font-size="11" font-weight="bold" fill="#856404">Consumer</text>
  <text x="640" y="122" text-anchor="middle" font-size="10" fill="#856404">take(8, pipeline)</text>
  <text x="640" y="138" text-anchor="middle" font-size="9"  fill="#856404">[1, 9, 25, 49 &#8230;]</text>
  <line x1="148" y1="100" x2="194" y2="100" stroke="#3a8c3f" stroke-width="1.8" marker-end="url(#pp1)"/>
  <line x1="328" y1="100" x2="374" y2="100" stroke="#3a8c3f" stroke-width="1.8" marker-end="url(#pp1)"/>
  <line x1="508" y1="100" x2="554" y2="100" stroke="#3a8c3f" stroke-width="1.8" marker-end="url(#pp1)"/>
  <text x="171" y="94" text-anchor="middle" font-size="9" fill="#3a8c3f">values</text>
  <text x="351" y="94" text-anchor="middle" font-size="9" fill="#3a8c3f">values</text>
  <text x="531" y="94" text-anchor="middle" font-size="9" fill="#3a8c3f">values</text>
  <line x1="554" y1="140" x2="510" y2="140" stroke="#c0392b" stroke-width="1.8" marker-end="url(#pp2)"/>
  <line x1="374" y1="140" x2="330" y2="140" stroke="#c0392b" stroke-width="1.8" marker-end="url(#pp2)"/>
  <line x1="194" y1="140" x2="150" y2="140" stroke="#c0392b" stroke-width="1.8" marker-end="url(#pp2)"/>
  <text x="531" y="155" text-anchor="middle" font-size="9" fill="#c0392b">next()</text>
  <text x="351" y="155" text-anchor="middle" font-size="9" fill="#c0392b">next()</text>
  <text x="171" y="155" text-anchor="middle" font-size="9" fill="#c0392b">next()</text>
  <rect x="16" y="178" width="708" height="66" rx="8" fill="#e8ecf5" stroke="#888" stroke-width="1.2"/>
  <text x="370" y="198" text-anchor="middle" font-size="11" font-weight="bold" fill="#333">Why this is memory-efficient</text>
  <text x="370" y="216" text-anchor="middle" font-size="10" fill="#555">Consumer pulls one value at a time (red). Each stage wakes, produces one item, then suspends again.</text>
  <text x="370" y="232" text-anchor="middle" font-size="10" fill="#555">Only one value exists in memory at any point &#8212; total memory stays O(1) regardless of pipeline length.</text>
</svg>
<figcaption style="font-size:0.85em;color:#555;margin-top:4px;text-align:center">
  Green arrows: value flows forward. Red arrows: <code>next()</code> pulls propagate backward through the chain.
</figcaption>
</figure>
"""))

In [27]:
# Advantage 5 & 6 — state management vs class-based iterator

# Class-based iterator: boilerplate-heavy
class RangeSquaresClass:
    def __init__(self, n):
        self.n = n
        self.i = 1
    def __iter__(self):
        return self
    def __next__(self):
        if self.i > self.n:
            raise StopIteration
        val = self.i * self.i
        self.i += 1
        return val


# Generator equivalent: 2 lines
def range_squares_gen(n):
    for i in range(1, n + 1):
        yield i * i


print("Class  :", list(RangeSquaresClass(6)))
print("Generator:", list(range_squares_gen(6)))
print("Results are identical — generator is far less code")

Class  : [1, 4, 9, 16, 25, 36]
Generator: [1, 4, 9, 16, 25, 36]
Results are identical — generator is far less code


## Disadvantages

| # | Disadvantage | Consequence |
|---|---|---|
| 1 | **Single-pass / one-shot** | Once a value is consumed it is gone; you cannot restart or rewind without recreating the generator |
| 2 | **No random access** | `gen[3]` raises `TypeError`; you must consume elements in order |
| 3 | **No `len()`** | Size is unknown until fully consumed; `len(gen)` raises `TypeError` |
| 4 | **Hard to debug** | Lazy execution means errors surface at consumption time, not at generator creation |
| 5 | **Not reusable by default** | The same generator object cannot be iterated more than once; each consumer needs a fresh call to the generator function |
| 6 | **Cannot look ahead / peek** | No built-in way to inspect the next value without consuming it (requires `itertools.islice` or a wrapper) |

In [28]:
# Disadvantage 1 — single-pass: exhausted generators produce nothing
gen = squares(5)

print("First pass :", list(gen))     # consumes the generator
print("Second pass:", list(gen))     # empty — already exhausted
print()

# Fix: call the generator function again
print("Fresh generator:", list(squares(5)))

First pass : [1, 4, 9, 16, 25]
Second pass: []

Fresh generator: [1, 4, 9, 16, 25]


In [29]:
# Disadvantage 2 & 3 — no indexing, no len
gen = squares(5)

try:
    print(gen[2])
except TypeError as e:
    print(f"Indexing error : {e}")

try:
    print(len(gen))
except TypeError as e:
    print(f"len() error    : {e}")

print()
# Workaround: materialise to a list when you need indexing or length
values = list(squares(5))
print(f"Materialised list: {values},  len={len(values)},  index[2]={values[2]}")

Indexing error : 'generator' object is not subscriptable
len() error    : object of type 'generator' has no len()

Materialised list: [1, 4, 9, 16, 25],  len=5,  index[2]=9


In [30]:
# Disadvantage 4 — errors surface at consumption, not at creation
def risky_gen(data):
    for item in data:
        yield int(item)          # will fail when item is not numeric


gen = risky_gen(["1", "2", "oops", "4"])   # no error yet
print("Generator created without error")

try:
    result = list(gen)           # error only now, during consumption
except ValueError as e:
    print(f"Error during consumption: {e}")

print()

# Disadvantage 5 — not reusable (only the function is reusable, not the object)
gen_obj = squares(3)             # this specific object is one-shot
list(gen_obj)                    # consume it
print("Reusing same object:", list(gen_obj))  # empty
print("New generator call :", list(squares(3)))  # fresh

print()

# Disadvantage 6 — no peek: use itertools or a peekable wrapper
import itertools
gen = squares(5)
first, gen = itertools.tee(gen, 2)       # tee allows peeking but stores values
peek = next(first)
print(f"Peeked at first value: {peek}")
print(f"Main generator still yields from start: {list(gen)}")

Generator created without error
Error during consumption: invalid literal for int() with base 10: 'oops'

Reusing same object: []
New generator call : [1, 4, 9]

Peeked at first value: 1
Main generator still yields from start: [1, 4, 9, 16, 25]


## Summary

| | Generator | List |
|---|---|---|
| Memory | O(1) — one value at a time | O(n) — all values at once |
| Evaluation | Lazy — on demand | Eager — upfront |
| Infinite sequences | Yes | No |
| Random access `[i]` | No | Yes |
| `len()` | No | Yes |
| Reusable | No (re-call the function) | Yes |
| Pipeline composition | Natural | Requires intermediate lists |

**Rule of thumb:** use a generator when you process data **once, in order**, and the dataset is **large or unbounded**.  
Materialise to a list only when you need **random access, repeated iteration, or `len`**.